In [2]:
import pandas as pd

# Table 8 (Appendix Table 10): Pre/Post Differences

In [3]:
def rescale(min_prev, max_prev, min_new, max_new, value):
    return (value - min_prev) / (max_prev - min_prev) * (max_new - min_new) + min_new

In [4]:
pre_post = pd.read_csv('./data/pre_post.csv')

for col in pre_post.columns:
    # _norm cols are [0,1] min-max scales
    # scale the scores to a 5pt Likert scale (1-5)
    if '_norm' in col:
        pre_post[col] = pre_post[col].apply(lambda x: rescale(0, 1, 1, 5, x))

    if 'satisfaction' in col:
        # satisfaction scores are on a 7pt scale (1-7)
        # scale the satisfaction scores to a 5pt Likert scale (1-5)
        pre_post[col] = pre_post[col].apply(lambda x: rescale(1, 7, 1, 5, x))

    if "exercise_past_7_" in col:
        # exercise scores are on a 4pt scale (1-4)
        # scale the exercise scores to a 5pt Likert scale (1-5)
        pre_post[col] = pre_post[col].apply(lambda x: rescale(1, 4, 1, 5, x))

In [5]:
import numpy as np

def mean_sd(series):
    a = series.dropna().astype(float)
    return a.mean(), a.std(ddof=1)

def diff_in_diff_se(treat_diff, ctrl_diff):
    nt, nc = len(treat_diff.dropna()), len(ctrl_diff.dropna())
    vt, vc = treat_diff.var(ddof=1), ctrl_diff.var(ddof=1)
    return np.sqrt(vt / nt + vc / nc)

def fmt(m, sd):          # mean (SD) with 2 decimals
    return f'{m:.2f} ({sd:.2f})'

In [6]:
BASE_METRICS = [
    ('Stage of change',                'stage_of_change_score'),
    ('IPAQ  MET‑min',                  'ipaq_met'),
    ('IPAQ  level',                    'ipaq_level_int'),

    # Physical Activity & Health (10 lines)
    ('General Health',                 'general_health'),
    ('Importance of Health',           'importance_health'),
    ('Physical Fitness',               'physical_fitness'),
    ('Exercise Past 7 d',              'exercise_past_7'),

    # Satisfaction (4)
    ('Satisfaction: PA‑level',         'satisfaction_physical_activity'),
    ('Satisfaction: Phys Health',      'satisfaction_physical_health'),
    ('Satisfaction: Mental Health',    'satisfaction_mental_health'),
    ('Satisfaction: Life',             'satisfaction_life'),

    # Motivation (2)
    ('Intention to increase PA',       'intention_increase_pa'),
    ('Motivation to be healthy',       'motivation_health'),
]

# # 16 pre/post UX items
ux_prefixes = [
    'personalized', 'actionable', 'generic', 'comfortable', 'supported',
    'capable', 'motivated', 'opinion', 'understood', 'unsolicited',
    'empathetic', 'relevant', 'obstacles', 'reflect', 'specific', 'insights'
]
BASE_METRICS += [(f'UX: {p.replace("_", " ").title()}', p) for p in ux_prefixes]

# Summated scales – we will *expand* each prefix to grab every column
SCALE_PREFIXES = [
    ('PA & Health',            'pa_health_'),
    ('UX & Advice',            'ux_'),
    ('Self‑Efficacy',          'self_efficacy_'),
    ('PA Adequacy Mindset',    'adequacy_mindset_'),
    ('Exercise‑Process Mindset','process_mindset_exercise_'),
    ('Health‑Process Mindset', 'process_mindset_health_'),
    ('Barriers to Being Active','barriers_'),
    ('SUS',                    'sus_'),
    ('SASSI',                  'sassi_'),
    ('TSRI',                   'tsri_'),
    ('eHealth Literacy',       'ehealth_literacy_'),
    ('Shared Decision Making', 'sdmq9_'),
]

# Post‑only UX extras
POST_ONLY = [
    ('Feels human‑like',          'human_like_post'),
    ('Wellbeing cared for',       'wellbeing_cared_for_post'),
    ('Beebo cared',               'beebo_cared_post'),
    ('Anthropomorphism',          'anthropomorphism_post'),
    ('Anthropomorphism: norm',  'anthropomorphism_norm_post'),
]

In [12]:
rows = []

treat = pre_post[pre_post['condition'] == 'T']
ctrl  = pre_post[pre_post['condition'] == 'C']

def add_metric(label, base):
    pre_col, post_col = f'{base}_pre', f'{base}_post'
    if pre_col not in pre_post.columns or post_col not in pre_post.columns:
        # silently skip if column is missing (e.g., post‑only extras)
        return
    t_pre_m, t_pre_se = mean_sd(treat[pre_col])
    t_post_m,t_post_se= mean_sd(treat[post_col])
    c_pre_m, c_pre_se = mean_sd(ctrl[pre_col])
    c_post_m,c_post_se= mean_sd(ctrl[post_col])

    t_diff = treat[post_col] - treat[pre_col]
    c_diff = ctrl [post_col] - ctrl [pre_col]

    dt_m, dt_se = mean_sd(t_diff)
    dc_m, dc_se = mean_sd(c_diff)

    dnd_m = dt_m - dc_m

    rows.append({
        'Measure': label,
        'T pre':   fmt(t_pre_m, t_pre_se),
        'T post':  fmt(t_post_m,t_post_se),
        'ΔT':      fmt(dt_m,    dt_se),
        'C pre':   fmt(c_pre_m, c_pre_se),
        'C post':  fmt(c_post_m,c_post_se),
        'ΔC':      fmt(dc_m,    dc_se),
        'ΔΔ':      dnd_m
    })

def add_post_only(label, col):
    t_m, t_se = mean_sd(treat[col])
    c_m, c_se = mean_sd(ctrl[col])

    rows.append({
        'Measure': label,
        'T pre':   '',
        'T post':  fmt(t_m, t_se),
        'ΔT':      '',
        'C pre':   '',
        'C post':  fmt(c_m, c_se),
        'ΔC':      '',
        'ΔΔ':      '',
    })

for lab, base in BASE_METRICS:
    add_metric(lab, base)

for name, prefix in SCALE_PREFIXES:
    # collect all “stem” names *without* _pre/_post
    stems = sorted(set(c[:-4] for c in pre_post.columns if c.startswith(prefix) and c.endswith('_pre')))
    for s in stems:       # total, total_norm, subscales, items – everything
        label = f'{name}: {s[len(prefix):]}' if s != prefix.rstrip('_') else name
        add_metric(label, s)

# --- post‑only UX extras
for lab, col in POST_ONLY:
    add_post_only(lab, col)

out = pd.DataFrame(rows,
                   columns=['Measure','T pre','T post','ΔT',
                            'C pre','C post','ΔC', 'ΔΔ'])

pd.set_option('display.max_rows', None, 'display.max_colwidth', None)

print('\n' + '='* 160)
print(out.to_string(index=False))
print('=' * 160)


                                    Measure           T pre          T post              ΔT           C pre          C post              ΔC         ΔΔ
                            Stage of change     1.88 (0.59)     2.85 (0.73)     0.96 (0.96)     1.89 (0.74)     2.61 (1.07)     0.71 (1.01)   0.247253
                              IPAQ  MET‑min 415.31 (329.00) 749.73 (467.88) 334.42 (505.20) 548.18 (637.10) 893.96 (741.87) 345.79 (965.50) -11.362637
                                IPAQ  level     1.27 (0.45)     1.77 (0.51)     0.50 (0.65)     1.25 (0.44)     1.68 (0.67)     0.43 (0.79)   0.071429
                             General Health     2.81 (0.69)     3.15 (0.61)     0.35 (0.49)     2.75 (0.80)     2.93 (0.81)     0.18 (0.55)   0.167582
                       Importance of Health     4.04 (0.60)     4.15 (0.61)     0.12 (0.52)     3.71 (0.76)     3.86 (0.85)     0.14 (0.71)  -0.027473
                           Physical Fitness     2.12 (0.77)     2.38 (0.80)     0.27 (0.67)  

# Daily Data

In [8]:
daily = pd.read_csv('./data/daily.csv')

In [9]:
DAILY_QS = ['I am satisfied with my current level of physical activity._num',
            'I am a physically active person._num',
            'I am committed to my physical activity goal._num',
            'How hopeful are you about your physical health today?_num',
            'How was your mood today?_num']

daily = daily.dropna(subset=DAILY_QS)

daily_t = daily[daily['Condition'] == 'T']
daily_c = daily[daily['Condition'] == 'C']

rows = []

for q in DAILY_QS:
    # Day 1 and Day 28 subsets
    day1_t = daily_t[daily_t['Day'] == 0][q]
    day28_t = daily_t[daily_t['Day'] == 27][q]

    day1_c = daily_c[daily_c['Day'] == 0][q]
    day28_c = daily_c[daily_c['Day'] == 27][q]

    # Linear regression: score ~ day
    slope_t, intercept_t = np.polyfit(daily_t['Day'], daily_t[q], 1)
    slope_c, intercept_c = np.polyfit(daily_c['Day'], daily_c[q], 1)

    # New Δ definition: predicted change from Day 0 to Day 28
    delta_t = 28 * slope_t
    delta_c = 28 * slope_c

    rows.append({
        'Question': q[:-4],
        'Mean (T)': fmt(*mean_sd(daily_t[q])),
        'Day 1 (T)': fmt(*mean_sd(day1_t)),
        'Day 28 (T)': fmt(*mean_sd(day28_t)),
        'ΔT': f"{delta_t:.2f}",
        'Slope (T)': slope_t,
        'Intercept (T)': intercept_t,
        'Mean (C)': fmt(*mean_sd(daily_c[q])),
        'Day 1 (C)': fmt(*mean_sd(day1_c)),
        'Day 28 (C)': fmt(*mean_sd(day28_c)),
        'ΔC': f"{delta_c:.2f}",
        'Slope (C)': slope_c,
        'Intercept (C)': intercept_c
    })

out = pd.DataFrame(rows)
pd.set_option('display.max_rows', None, 'display.max_colwidth', None)

print('\n' + '='* 160)
print(out.to_string(index=False))
print('=' * 160)


                                                  Question    Mean (T)   Day 1 (T)  Day 28 (T)    ΔT  Slope (T)  Intercept (T)    Mean (C)   Day 1 (C)  Day 28 (C)    ΔC  Slope (C)  Intercept (C)
I am satisfied with my current level of physical activity. 3.17 (1.09) 2.59 (1.18) 3.45 (1.01)  0.62   0.022180       2.864675 3.06 (1.07) 2.71 (1.10) 2.91 (1.16)  0.17   0.006069       2.971772
                          I am a physically active person. 3.15 (1.03) 2.77 (1.02) 3.32 (0.95)  0.84   0.030049       2.729483 3.02 (1.05) 2.53 (0.87) 3.00 (1.00)  0.25   0.008837       2.887318
              I am committed to my physical activity goal. 4.27 (0.66) 4.23 (0.53) 4.09 (0.81) -0.21  -0.007444       4.372116 3.96 (0.84) 3.82 (0.73) 3.91 (0.90) -0.01  -0.000471       3.971386
     How hopeful are you about your physical health today? 4.07 (0.70) 3.91 (0.68) 4.14 (0.71)  0.14   0.005120       3.998105 3.84 (0.85) 3.88 (0.60) 3.87 (0.87) -0.09  -0.003071       3.887714
                        

# Weekly Data

In [10]:
weekly = pd.read_csv('./data/weekly.csv')

In [11]:
WEEKLY_QS = ['overall_rating_num', 'wallpaper_rating_num', 'mindset_ease_num', 'mindset_fun_num', 'barrier_total']

weekly = weekly.dropna(subset=WEEKLY_QS)

# rescale
weekly['overall_rating_num']   = weekly['overall_rating_num'].apply(lambda x: rescale(1, 10, 1, 5, x))
weekly['wallpaper_rating_num'] = weekly['wallpaper_rating_num'].apply(lambda x: rescale(1, 10, 1, 5, x))
weekly['mindset_ease_num']     = weekly['mindset_ease_num'].apply(lambda x: rescale(1, 4, 1, 5, x))
weekly['mindset_fun_num']      = weekly['mindset_fun_num'].apply(lambda x: rescale(1, 4, 1, 5, x))
weekly['barrier_total']        = weekly['barrier_total'].apply(lambda x: rescale(0, 12, 1, 5, x))

weekly_t = weekly[weekly['Condition'] == 'T']
weekly_c = weekly[weekly['Condition'] == 'C']

rows = []

for q in WEEKLY_QS:
    # Week 1 and Week 4 subsets
    week1_t = weekly_t[weekly_t['Week'] == 1][q]
    week4_t = weekly_t[weekly_t['Week'] == 4][q]

    week1_c = weekly_c[weekly_c['Week'] == 1][q]
    week4_c = weekly_c[weekly_c['Week'] == 4][q]

    # Linear regression: score ~ week
    slope_t, intercept_t = np.polyfit(weekly_t['Week'], weekly_t[q], 1)
    slope_c, intercept_c = np.polyfit(weekly_c['Week'], weekly_c[q], 1)

    # New Δ definition: predicted change across 4 weeks
    delta_t = 4 * slope_t
    delta_c = 4 * slope_c

    rows.append({
        'Question': q[:-4],  # strip "_num"
        'Mean (T)':   fmt(*mean_sd(weekly_t[q])),
        'Week 1 (T)': fmt(*mean_sd(week1_t)),
        'Week 4 (T)': fmt(*mean_sd(week4_t)),
        'β_T':        f"{slope_t:.3f}",
        'ΔT':         f"{delta_t:.2f}",
        'Mean (C)':   fmt(*mean_sd(weekly_c[q])),
        'Week 1 (C)': fmt(*mean_sd(week1_c)),
        'Week 4 (C)': fmt(*mean_sd(week4_c)),
        'β_C':        f"{slope_c:.3f}",
        'ΔC':         f"{delta_c:.2f}",
        'Slope (T)':  slope_t,
        'Intercept (T)': intercept_t,
        'Slope (C)':  slope_c,
        'Intercept (C)': intercept_c,
    })

out = pd.DataFrame(rows)
pd.set_option('display.max_rows', None, 'display.max_colwidth', None)

print('\n' + '='*160)
print(out.to_string(index=False))
print('='*160)


        Question    Mean (T)  Week 1 (T)  Week 4 (T)    β_T    ΔT    Mean (C)  Week 1 (C)  Week 4 (C)    β_C    ΔC  Slope (T)  Intercept (T)  Slope (C)  Intercept (C)
  overall_rating 3.84 (0.74) 3.90 (0.67) 3.98 (0.81) -0.005 -0.02 4.02 (0.69) 3.94 (0.68) 4.02 (0.68)  0.040  0.16  -0.005080       3.853377   0.039627       3.927021
wallpaper_rating 3.78 (1.12) 3.70 (1.18) 3.93 (1.22)  0.077  0.31 3.94 (0.94) 3.73 (0.98) 4.14 (0.81)  0.153  0.61   0.076726       3.601347   0.152578       3.590734
    mindset_ease 3.11 (0.82) 2.92 (0.78) 3.35 (1.11)  0.139  0.56 2.93 (1.00) 2.90 (1.17) 2.87 (0.98)  0.024  0.10   0.139314       2.785206   0.024204       2.870796
     mindset_fun 3.17 (0.91) 3.13 (0.86) 3.27 (1.03)  0.045  0.18 2.96 (1.06) 2.81 (1.10) 2.87 (1.10)  0.056  0.22   0.044713       3.067285   0.055526       2.829080
       barrier_t 2.54 (0.70) 2.51 (0.74) 2.53 (0.71) -0.009 -0.04 2.79 (0.85) 2.92 (0.86) 2.60 (0.96) -0.059 -0.24  -0.008986       2.556682  -0.059085       2.9271